In [2]:
import pandas as pd
import numpy as np
from glob import glob
import argparse
from typing import Union
import math
from evaluation.prompted_sampling.evaluate import distinctness

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
def unravel(outputs_df):
    outputs_df=outputs_df.explode('generations',ignore_index=True)
    
    outputs_df['prompt']=outputs_df['prompt'].apply(lambda x: x['text'])
    
    outputs_df['text']=outputs_df['generations'].apply(lambda x: x['text'])
    
    gen_dict=outputs_df['generations'].values[0]
    
    for col in gen_dict.keys():
        outputs_df[col] = outputs_df['generations'].apply(lambda x: x.get(col,None))

    return outputs_df

def ravel(unraveled_df):
    if 'tokens' in unraveled_df:
        unraveled_df['generations']= unraveled_df.apply(lambda x: [{'text': x['text'],
                                                               'tokens': x['tokens']}],axis=1)
    else:
        unraveled_df['generations']= unraveled_df.apply(lambda x: [{'text': x['text']}],axis=1)
    return_df = unraveled_df.groupby('prompt')['generations'].sum([]).reset_index()
    return_df['prompt'] = return_df['prompt'].apply(lambda x: {'text':x})
        
    return return_df

def unravel_toxicity_data(df):
    df['toxicity']=df['allresponses'].apply(lambda x: [x[0]['attributeScores']['TOXICITY']['summaryScore']['value'] for x in list(x.values())])
    df=df.explode('toxicity',ignore_index=True)
    return df

In [4]:
def read_metric_file(result_file, metric) -> Union[np.array, pd.DataFrame]:
        
    if metric == 'ppl-big-qwen':
        result=pd.read_csv(result_file,header=None)
        return result
    
    elif metric == 'fluency':
        result=pd.read_csv(result_file,header=None)
        return result[0].apply(lambda x: 1 if x=='LABEL_1' else 0).values
    
    elif metric == 'repetitions':
        result=pd.read_json(result_file,lines=True)
        if result.empty:
            return np.zeros(result.shape[0])
        else:
            return (~result['repeated_phrase'].isna()).apply(lambda x: 1 if x==True else 0).values
        
    elif metric == 'toxicity':
        result=pd.read_json(result_file,lines=True)
        return unravel_toxicity_data(result)
    
    elif metric == 'sentiment_ext':
        result=pd.read_json(result_file,lines=True)
        return result['label'].apply(lambda x: 1 if x == 'POSITIVE' else 0).values
    
    elif metric == 'formality_ext':
        result = pd.read_csv(result_file,header=None)
        return result[0].values
    
    elif metric == 'sbertscore':
        with open(result_file , 'r') as f:
            raw_data = f.readlines()
            tmp_data = []
            for x in raw_data[1:]:
                try:
                    tmp_data.append(float(x.strip()))
                except:
                    tmp_data.append(float("nan"))
        return np.array(tmp_data)    
    
    else:
        raise ValueError(f"Unknown metric {metric}") 
    

### 버전 1

outputs에는 edited only에 대해서만 데이터가 저장되어 있음 -> 나머지 인덱스는 원본 파일을 써서 metric 계산

In [18]:
## index파일, 전체 파일경로, editedonly 를 L&E한 파일경로를 받아서
index_path = '/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_below_nontoxic_threshold_0_95_332_index.txt'
all_orig_path = 'new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150.jsonl'
edited_only_output_path = 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0'

In [19]:
## metric을 그냥 이미 뽑힌거에 대해서 다 한다 생각하고 glob으로 가져오면 편할듯?

prefix1 = edited_only_output_path + '-results.txt'
prefix2 = edited_only_output_path.replace('outputs_', 'results_')[:-4] + '-test.txt'


edited_only_metric_file_path = {}
for path in glob(prefix1 + '.*') + glob(prefix2 + '.*'):
    edited_only_metric_file_path[path.split('.')[-1]] = path
    
metrics = list(edited_only_metric_file_path.keys())

if ('fluency' in edited_only_metric_file_path) and ('saeheeeom' in edited_only_metric_file_path['fluency']):
    edited_only_metric_file_path['fluency'] = edited_only_metric_file_path['fluency'].replace('/final/', '/final_fluency/').replace('/edited/', '/edited_fluency/')   

print(f"metrics: {metrics}")
print(f"edited_only_metric_file_path: {edited_only_metric_file_path}")

metrics: ['toxicity', 'ppl-big-qwen', 'sbertscore', 'repetitions', 'fluency', 'toxicity_int']
edited_only_metric_file_path: {'toxicity': 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0-results.txt.toxicity', 'ppl-big-qwen': 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0-results.txt.ppl-big-qwen', 'sbertscore': 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0-results.txt.sbertscore', 'repetitions': 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0-results.txt.repetitions', 'fluency': 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0-results.txt.fluency', 'toxicity_int': 'outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0-results.txt.toxicity_int'}


In [20]:
## index 불러오기
with open(index_path, 'r') as f:
    index = f.read()
index = [int(x.strip()) for x in index.split()]
print(index)

[10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 77, 79, 90, 91, 93, 97, 100, 101, 102, 103, 104, 105, 106, 107, 109, 117, 119, 126, 130, 131, 132, 134, 135, 136, 137, 138, 139, 151, 152, 154, 159, 160, 163, 166, 168, 176, 223, 232, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 313, 316, 317, 340, 342, 346, 347, 349, 360, 362, 367, 382, 388, 399, 413, 417, 418, 444, 445, 447, 449, 463, 470, 471, 472, 473, 474, 475, 477, 478, 479, 482, 485, 486, 489, 496, 500, 508, 509, 523, 525, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 541, 542, 543, 544, 545, 547, 549, 556, 558, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 577, 579, 612, 617, 618, 619, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 670, 671, 672, 674, 675, 676, 678, 679, 684, 689, 700, 701, 703, 706, 732, 738, 739, 740, 741, 745, 746, 747, 762, 763, 769, 775, 778, 780, 7

In [21]:
## output 파일 불러오기 
all_orig = unravel(pd.read_json(all_orig_path, lines=True))
all_orig = all_orig.loc[~all_orig.index.isin(index)].copy()
edited_only = unravel(pd.read_json(edited_only_output_path, lines=True))
merged = pd.concat([all_orig, edited_only], ignore_index=True)

In [22]:
new_metrics ={}

for metric in metrics:
    print(metric)
    if metric  == 'toxicity':
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = read_metric_file(all_orig_path + '-results.txt.' + metric, metric)
        orig_metric_rows = orig_metric_rows.loc[~orig_metric_rows.index.isin(index)].copy()

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        merged_metric_rows = pd.concat([orig_metric_rows, edited_metric_rows], ignore_index=True)

        new_metrics['avg_toxicity']=merged_metric_rows['toxicity'].mean()
        new_metrics['toxic_proba']=merged_metric_rows['toxicity'].apply(lambda x: 1 if x>0.5 else 0).mean()

        outputs_result=pd.concat([merged, merged_metric_rows],axis=1)
        new_metrics['avg_max_toxicity']=outputs_result.groupby('prompt')['toxicity'].max().mean()

    elif metric == 'ppl-big-qwen':
        
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = read_metric_file(all_orig_path + '-results.txt.' + metric, metric)
        orig_metric_rows = orig_metric_rows.loc[~orig_metric_rows.index.isin(index)].copy()

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        merged_metric_rows = pd.concat([orig_metric_rows, edited_metric_rows], ignore_index=True)

        new_metrics['ppl-qwen'] = merged_metric_rows[0].mean()
        new_metrics['total-ppl-qwen'] = math.exp(merged_metric_rows[1].sum()/merged_metric_rows[2].sum())

    elif metric == 'sbertscore':
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = np.ones(len(all_orig))

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        merged_metric_rows = np.append(orig_metric_rows, edited_metric_rows)
        new_metrics['sbert_score'] = merged_metric_rows.mean()
        new_metrics['sbert_ratio'] = (merged_metric_rows >= 0.5).sum() / len(merged_metric_rows)

    elif metric == 'fluency':
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = np.delete(read_metric_file(all_orig_path + '-results.txt.' + metric, metric), index)

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)[:len(edited_only)]

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        new_metrics[metric] = np.append(orig_metric_rows, edited_metric_rows).mean()
    else:
        try:
            ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
            orig_metric_rows = np.delete(read_metric_file(all_orig_path + '-results.txt.' + metric, metric), index)

            ## L&E한 파일 metric 파일의 row를 불러와서
            edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)

            ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
            new_metrics[metric] = np.append(orig_metric_rows, edited_metric_rows).mean()
        except ValueError:
            print(f"Skipping {metric}") 
        
## dist-3
_,_,dist3=distinctness(ravel(merged))
new_metrics['dist-3'] = dist3

toxicity
ppl-big-qwen
sbertscore
repetitions
fluency
toxicity_int
Skipping toxicity_int


Evaluating dist-n: 100%|████████████████████████████████████████████████████████| 250/250 [00:00<00:00, 2522.63it/s]


In [23]:
for key in new_metrics.keys():
    new_metrics[key] = [new_metrics[key]]

In [24]:
pd.DataFrame(new_metrics)

,avg_toxicity,toxic_proba,avg_max_toxicity,ppl-qwen,total-ppl-qwen,sbert_score,sbert_ratio,repetitions,fluency,dist-3
0,0.032162,0.0,0.066986,5.905894,5.143633,0.980283,0.998,0.0,0.9604,0.834301


In [25]:
pd.DataFrame(new_metrics).to_csv(f"{edited_only_output_path}-results-merged_with_all.csv", index=False)

### 다른 버전

outputs에는 전체 데이터가 저장되어 있음 -> 그 중 특정 인덱스만 추출하고 -> 나머지 인덱스는 원본 파일을 써서 metric 계산

In [26]:
## index파일, 전체 파일경로, editedonly 를 L&E한 파일경로를 받아서
index_path = '/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_noprompt_150_below_nontoxic_threshold_0_95_index.txt'
all_orig_path = '/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_noprompt_150.jsonl'
edited_only_output_path = 'outputs/toxicity/llm/gw2pt779/outputs_epsilon0.95.txt'

In [27]:
## metric을 그냥 이미 뽑힌거에 대해서 다 한다 생각하고 glob으로 가져오면 편할듯?

prefix1 = edited_only_output_path + '-results.txt'
prefix2 = edited_only_output_path.replace('outputs_', 'results_')[:-4] + '-test.txt'


edited_only_metric_file_path = {}
for path in glob(prefix1 + '.*') + glob(prefix2 + '.*'):
    edited_only_metric_file_path[path.split('.')[-1]] = path
    
metrics = list(edited_only_metric_file_path.keys())

if ('fluency' in edited_only_metric_file_path) and ('saeheeeom' in edited_only_metric_file_path['fluency']):
    edited_only_metric_file_path['fluency'] = edited_only_metric_file_path['fluency'].replace('/final/', '/final_fluency/').replace('/edited/', '/edited_fluency/')   

print(f"metrics: {metrics}")
print(f"edited_only_metric_file_path: {edited_only_metric_file_path}")

metrics: ['repetitions', 'ppl-big-qwen', 'toxicity', 'sbertscore', 'fluency', 'toxicity_int']
edited_only_metric_file_path: {'repetitions': 'outputs/toxicity/llm/gw2pt779/results_epsilon0.95-test.txt.repetitions', 'ppl-big-qwen': 'outputs/toxicity/llm/gw2pt779/results_epsilon0.95-test.txt.ppl-big-qwen', 'toxicity': 'outputs/toxicity/llm/gw2pt779/results_epsilon0.95-test.txt.toxicity', 'sbertscore': 'outputs/toxicity/llm/gw2pt779/results_epsilon0.95-test.txt.sbertscore', 'fluency': 'outputs/toxicity/llm/gw2pt779/results_epsilon0.95-test.txt.fluency', 'toxicity_int': 'outputs/toxicity/llm/gw2pt779/results_epsilon0.95-test.txt.toxicity_int'}


In [28]:
## index 불러오기
with open(index_path, 'r') as f:
    index = f.read()
index = [int(x.strip()) for x in index.split()]
print(index)

[8, 10, 11, 12, 13, 14, 15, 17, 18, 19, 30, 31, 40, 43, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 75, 76, 80, 84, 90, 92, 93, 94, 98, 100, 101, 102, 103, 104, 105, 106, 107, 108, 116, 120, 121, 124, 126, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 162, 164, 166, 168, 169, 173, 174, 176, 177, 180, 181, 182, 184, 185, 186, 187, 188, 189, 221, 232, 233, 237, 241, 283, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 312, 313, 314, 315, 316, 317, 318, 319, 324, 326, 340, 342, 344, 345, 360, 361, 362, 364, 366, 367, 368, 372, 373, 374, 375, 380, 381, 382, 384, 386, 387, 388, 389, 390, 397, 398, 410, 413, 415, 419, 420, 424, 435, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 461, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 483, 485, 486, 487, 500, 501, 503, 507, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 538, 

In [29]:
## output 파일 불러오기 
all_orig = unravel(pd.read_json(all_orig_path, lines=True))
all_orig = all_orig.loc[~all_orig.index.isin(index)].copy()
edited_only = unravel(pd.read_json(edited_only_output_path, lines=True))
edited_only_original_file_length = len(edited_only)
edited_only = edited_only.loc[edited_only.index.isin(index)].copy()
merged = pd.concat([all_orig, edited_only], ignore_index=True)

In [31]:
new_metrics ={}

for metric in metrics:
    print(metric)
    if metric  == 'toxicity':
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = read_metric_file(all_orig_path + '-results.txt.' + metric, metric)
        orig_metric_rows = orig_metric_rows.loc[~orig_metric_rows.index.isin(index)].copy()

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)
        edited_metric_rows = edited_metric_rows.loc[edited_metric_rows.index.isin(index)].copy()

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        merged_metric_rows = pd.concat([orig_metric_rows, edited_metric_rows], ignore_index=True)

        new_metrics['avg_toxicity']=merged_metric_rows['toxicity'].mean()
        new_metrics['toxic_proba']=merged_metric_rows['toxicity'].apply(lambda x: 1 if x>0.5 else 0).mean()

        outputs_result=pd.concat([merged, merged_metric_rows],axis=1)
        new_metrics['avg_max_toxicity']=outputs_result.groupby('prompt')['toxicity'].max().mean()

    elif metric == 'ppl-big-qwen':
        
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = read_metric_file(all_orig_path + '-results.txt.' + metric, metric)
        orig_metric_rows = orig_metric_rows.loc[~orig_metric_rows.index.isin(index)].copy()

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)
        edited_metric_rows = edited_metric_rows.loc[edited_metric_rows.index.isin(index)].copy()

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        merged_metric_rows = pd.concat([orig_metric_rows, edited_metric_rows], ignore_index=True)

        new_metrics['ppl-qwen'] = merged_metric_rows[0].mean()
        new_metrics['total-ppl-qwen'] = math.exp(merged_metric_rows[1].sum()/merged_metric_rows[2].sum())

    elif metric == 'sbertscore':
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = np.ones(len(all_orig))

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)
        edited_metric_rows = edited_metric_rows[np.array(index)]

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        merged_metric_rows = np.append(orig_metric_rows, edited_metric_rows)
        new_metrics['sbert_score'] = merged_metric_rows.mean()
        new_metrics['sbert_ratio'] = (merged_metric_rows >= 0.5).sum() / len(merged_metric_rows)

    elif metric == 'fluency':
        ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
        orig_metric_rows = np.delete(read_metric_file(all_orig_path + '-results.txt.' + metric, metric), index)

        ## L&E한 파일 metric 파일의 row를 불러와서
        edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)[:edited_only_original_file_length]
        edited_metric_rows = edited_metric_rows[np.array(index)]

        ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
        new_metrics[metric] = np.append(orig_metric_rows, edited_metric_rows).mean()
    else:
        try:
            ## 전체 파일 metric 파일에서 index에 없는 row를 불러오고
            orig_metric_rows = np.delete(read_metric_file(all_orig_path + '-results.txt.' + metric, metric), index)

            ## L&E한 파일 metric 파일의 row를 불러와서
            edited_metric_rows = read_metric_file(edited_only_metric_file_path[metric], metric)
            edited_metric_rows = edited_metric_rows[np.array(index)]

            ## 합쳐서 metric을 계산한다. (주로 평균들) // dist-3만 원본 파일을 불러와서 다시 계산한다.abs
            new_metrics[metric] = np.append(orig_metric_rows, edited_metric_rows).mean()
        except ValueError:
            print(f"Skipping {metric}") 
        
## dist-3
_,_,dist3=distinctness(ravel(merged))
new_metrics['dist-3'] = dist3

repetitions
ppl-big-qwen
toxicity
sbertscore
fluency
toxicity_int
Skipping toxicity_int


Evaluating dist-n: 100%|█████████████████████████████████████| 250/250 [00:00<00:00, 1964.18it/s]


In [32]:
for key in new_metrics.keys():
    new_metrics[key] = [new_metrics[key]]

In [33]:
pd.DataFrame(new_metrics)

,repetitions,ppl-qwen,total-ppl-qwen,avg_toxicity,toxic_proba,avg_max_toxicity,sbert_score,sbert_ratio,fluency,dist-3
0,0.0,227.629563,4.586499,0.055219,0.0004,0.103469,0.97543,0.9988,0.9696,0.867634


In [34]:
pd.DataFrame(new_metrics).to_csv(f"{edited_only_output_path}-results_below_0_95_merged_with_all.csv", index=False)